<a href="https://colab.research.google.com/github/gryfklapryd/data-science-2024/blob/main/Pertemuan3_Greyfikal_Apriyuda_250401020064.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama Lengkap: Greyfikal Apriyuda <br>
NIM: 250401020064 <br>
Kelas: Data Science IF401

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mstats
import requests

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("housing_dirty.csv")
df.head()

Saving housing_dirty.csv to housing_dirty (1).csv


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


In [ ]:
print("INFO")
df.info()

print("\nDESCRIBE")
display(df.describe())

print("\nMISSING VALUES")
display(df.isnull().sum())

INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB

DESCRIBE


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000



MISSING VALUES


,0
id,0
luas_m2,18
harga_juta,17
kota,0
kamar,10
tahun_bangun,0
kondisi,0


In [ ]:
print("Duplikat sebelum:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
print("Duplikat sesudah:", df.duplicated().sum())

Duplikat sebelum: 0
Duplikat sesudah: 0


In [ ]:
df['kota'] = df['kota'].astype(str).str.strip().str.title()
df['kondisi'] = df['kondisi'].astype(str).str.strip().str.lower()

df[['kota','kondisi']].head()

,kota,kondisi
0,Jogja,baik
1,Medan,bagus
2,Depok,baik
3,Ygy,baik
4,Medan,sedang


In [ ]:
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(exclude=np.number).columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

df.isnull().sum()

/tmp/ipykernel_5926/1471239568.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipykernel_5926/1471239568.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try us

,0
id,0
luas_m2,0
harga_juta,0
kota,0
kamar,0
tahun_bangun,0
kondisi,0


In [ ]:
def remove_outliers_iqr(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    before = data.shape[0]
    data = data[(data[col] >= lower) & (data[col] <= upper)]
    after = data.shape[0]

    print(f"{col} outlier removed:", before - after)
    return data

df = remove_outliers_iqr(df, 'harga_juta')
df = remove_outliers_iqr(df, 'luas_m2')

harga_juta outlier removed: 3
luas_m2 outlier removed: 1


In [ ]:
print("Total missing:", df.isnull().sum().sum())
print("Total duplicate:", df.duplicated().sum())

Total missing: 0
Total duplicate: 0


In [ ]:
df.to_csv("housing_clean.csv", index=False)
print("File housing_clean.csv berhasil dibuat")
files.download("housing_clean.csv")

File housing_clean.csv berhasil dibuat


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
url_api = "https://jsonplaceholder.typicode.com/posts"
response = requests.get(url_api)

df_api = pd.DataFrame(response.json())
df_api.head()
df_api.to_csv("jsonplaceholder_posts.csv", index=False)

Apa yang dipelajari: Proses data cleaning end-to-end pakai Pandas di Google Colab meliputi hapus duplikat, standarisasi format teks (huruf besar/kecil dan spasi), imputasi missing values (median untuk angka, modus untuk teks), deteksi dan hapus outlier pakai metode IQR, export hasil ke CSV, serta cara narik data dari API via library requests untuk dijadikan dataframe<br>

Temuan utama: Pembersihan data harus disesuaikan dengan tipe datanya (numerik vs kategorik). Metode IQR terbukti praktis untuk mendeteksi batas bawah dan atas outlier. Selain itu, integrasi data dari sumber eksternal (API) ke format tabel tabular Pandas ternyata sangat simpel<br>

Pertanyaan yang muncul: Belum ada pertanyaan<br>